In [1]:
import tensorflow as tf
import matplotlib.pyplot as plt
import numpy as np
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Flatten, Dense, Conv2D, MaxPool2D, Activation,  Dropout,LSTM,Embedding,Layer, LSTM, Conv1D, MaxPool1D, TimeDistributed, Input, GRU
# from tensorflow.keras.utils import to_categorical
from tensorflow.keras.optimizers import SGD, Adam
from tensorflow.keras import metrics
from tensorflow.keras.callbacks import Callback
import matplotlib.pyplot as plt
from tensorflow.keras.models import Model, load_model
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from tensorflow.keras.utils import plot_model
from scipy import signal
from keras.layers.rnn.dropout_rnn_cell_mixin import DropoutRNNCellMixin
import keras_tuner as kt
import keras_lmu
from sklearn.model_selection import train_test_split
#import torch
import tensorflow.keras as keras
import tensorflow.keras.backend as keras_backend
import random
%matplotlib qt

In [2]:
# strain = []
# stress = []

training_data   = []
outt   = []

# for i in range(1,5):
    
#     for j in range(1,4):
        
#         for k in range(1,4):
            
#             for l in range(1,4):

#                 strain.append(np.genfromtxt(r'C:\Users\Tandale\sciebo\olddata\Phd\Writing\SLMU_paper\BVPs\Stress-Strain_Data_csv\BVP_1_GeomNL\strain_'+str(i)+str(j)+str(k)+str(l)+'.csv'))
#                 stress.append(np.genfromtxt(r'C:\Users\Tandale\sciebo\olddata\Phd\Writing\SLMU_paper\BVPs\Stress-Strain_Data_csv\BVP_1_GeomNL\stress_'+str(i)+str(j)+str(k)+str(l)+'.csv'))

for i in range(10):
    training_data.append(np.genfromtxt(r'C:\Users\Tandale\sciebo\olddata\Phd\Writing\Meta_Learning\d'+str(i+1)+'.csv', delimiter=',')[0:1000,:])

input_NN_scaled = np.array(training_data)[:,:,0:4]
output_NN_scaled = np.array(training_data)[:,:,4:5]

In [3]:
# input_NN             = np.array(strain)
# output_NN            = np.array(stress)

# input_NN_r            = np.array(strain).reshape(input_NN.shape[0]*input_NN.shape[1], input_NN.shape[2])
# output_NN_r           = np.array(stress).reshape(output_NN.shape[0]*output_NN.shape[1], output_NN.shape[2])


# scaler1             =   MinMaxScaler(feature_range=(0, 1))
# scaler2             =   MinMaxScaler(feature_range=(0, 1))

# input_NN_scaled     =   scaler1.fit_transform(input_NN_r).reshape(input_NN.shape)
# output_NN_scaled    =   scaler2.fit_transform(output_NN_r).reshape(output_NN.shape)



input_nn_train_inp, input_nn_validation_inp, input_nn_train_output, input_nn_validation_output = train_test_split(input_NN_scaled, output_NN_scaled, test_size=0.3, random_state=42)

In [4]:
input_nn_train_inp, input_nn_test_inp, input_nn_train_output, input_nn_test_output = train_test_split(input_nn_train_inp[:,:,:], input_nn_train_output[:,:,:], test_size=0.4, random_state=42)

In [5]:
def pseudo_derivative(v_scaled, dampening_factor):
    return tf.math.atan(tf.maximum(0.,1 - tf.abs(v_scaled))*dampening_factor)

def pseudo_derivative_(v_scaled, dampening_factor):
    
    return 1-tf.math.square(tf.math.tanh(tf.maximum(0.,1 - tf.abs(v_scaled))*dampening_factor))


@tf.custom_gradient
def SpikeFunction(v_scaled, dampening_factor):

    # This means: z = 1 if v > thr else 0
    z_ = tf.greater(v_scaled, 0.)
    z_ = tf.cast(z_, dtype=tf.float32)

    def grad(dy):
        # This is where we overwrite the gradient
        # dy = dE/dz (total derivative) is the gradient back-propagated from the loss down to z(t)
        dE_dz = dy
        dz_dv_scaled = pseudo_derivative(v_scaled, dampening_factor)
        dE_dv_scaled = dE_dz * dz_dv_scaled

        return [dE_dv_scaled, tf.zeros_like(dampening_factor)]

    return tf.identity(z_, name="SpikeFunction"), grad


# Most important block for making Spiking LMU units

class SLMUcell(DropoutRNNCellMixin, tf.keras.layers.Layer):
    def __init__(self, memory_d, order, 
                 theta, hidden_cell, trainable_theta=False, discretizer="zoh",
                 thr=0.03, tau=20, dampening_factor=0.3, kernel_initializer="glorot_uniform",
                 kernel_regularizer=None,
                recurrent_regularizer=None,
                 recurrent_initializer="orthogonal",bias_initializer="zeros", bias_regularizer = None
                ):
        super().__init__()
        #self.num_neurons      = num_neurons
        #self.state_size       = (num_neurons, num_neurons)
        self.decay            = tf.exp(-1 / tau)

        self.dampening_factor = dampening_factor
        self.thr              = thr
        
        self.memory_d         = memory_d
        self.order            = order
        self._init_theta      = theta
        self.hidden_cell      = hidden_cell
        self.discretizer      = discretizer
        self.kernel_initializer = kernel_initializer
        self.recurrent_initializer = recurrent_initializer
        self.kernel            = None
        self.recurrent_kernel  = None
        self.bias              = None
        self.theta_inv         = None
        self.A                 = None
        self.B                 = None
        self.bias_initializer  = bias_initializer
        self.kernel_regularizer= kernel_regularizer
        self.bias_regularizer   = bias_regularizer
    
        if self.hidden_cell is None:
            for conn in ("hidden_to_memory", "input_to_hidden"):
                if getattr(self, conn):
                    raise ValueError(f"{conn} must be False if hidden_cell is None")

            self.hidden_output_size = self.memory_d * self.order
            self.hidden_state_size = []
            

        elif hasattr(self.hidden_cell, "state_size"):
            self.hidden_output_size = self.hidden_cell.output_size
            self.hidden_state_size = self.hidden_cell.state_size
        else:
            # TODO: support layers that don't have the `units` attribute
            self.hidden_output_size = self.hidden_cell.units
            self.hidden_state_size = [self.hidden_cell.units]

        self.state_size = tf.nest.flatten(self.hidden_state_size) + [
            self.memory_d * self.order
        ]
        self.output_size = self.hidden_output_size
        
    
    def theta(self):
        """
        Value of the ``theta`` parameter.
        If ``trainable_theta=True`` this returns the trained value, not the
        initial value passed in to the constructor.
        """
        if self.built:
            return 1 / tf.keras.backend.get_value(self.theta_inv)

        return self._init_theta
        

        
    def _gen_AB(self):
        """Generates A and B matrices."""

        # compute analog A/B matrices
        Q = np.arange(self.order, dtype=np.float64)
        R = (2 * Q + 1)[:, None]
        j, i = np.meshgrid(Q, Q)
        A = np.where(i < j, -1, (-1.0) ** (i - j + 1)) * R
        B = (-1.0) ** Q[:, None] * R
        
        
        self._base_A = tf.constant(A.T, dtype=self.dtype)
        self._base_B = tf.constant(B.T, dtype=self.dtype)

        self.A, self.B = SLMUcell._cont2discrete_zoh(self._base_A/self._init_theta, self._base_B/self._init_theta)
        
        
    def _cont2discrete_zoh(A, B):
        

        # combine A/B and pad to make square matrix
        em_upper = tf.concat([A, B], axis=0)
        em = tf.pad(em_upper, [(0, 0), (0, B.shape[0])])

        # compute matrix exponential
        ms = tf.linalg.expm(em)

        # slice A/B back out of combined matrix
        discrt_A = ms[: A.shape[0], : A.shape[1]]
        discrt_B = ms[A.shape[0] :, : A.shape[1]]

        return discrt_A, discrt_B
    
    
    def build(self, input_shape):
        """
        Builds the cell.
        Notes
        -----
        This method should not be called manually; rather, use the implicit layer
        callable behaviour (like ``my_layer(inputs)``), which will apply this method
        with some additional bookkeeping.
        """

        super().build(input_shape)
        
        enc_d   = input_shape[-1]
        
        if self.kernel_initializer is not None:
            self.kernel = self.add_weight(
                name="kernel",
                shape=(enc_d, self.memory_d),
                initializer=self.kernel_initializer,
                regularizer=self.kernel_regularizer,
            )
        
        elif enc_d != self.memory_d:
            raise ValueError(
                f"For LMUCells with no input kernel, the input dimension ({enc_d})"
                f" must equal `memory_d` ({self.memory_d})."
            )
    
        self.theta_inv = tf.constant(1 / self._init_theta, dtype=self.dtype)
        
        self.recurrent_kernel = None
        
        if self.hidden_cell is not None and not self.hidden_cell.built:
            
            hidden_input_d = self.memory_d * self.order
            
            with tf.name_scope(self.hidden_cell.name):
                self.hidden_cell.build((input_shape[0], hidden_input_d))
                
        self._gen_AB()
        
        
        
    def call(self, inputs, states, training=None):  # noqa: C901
        """
        Apply this cell to inputs.
        Notes
        -----
        This method should not be called manually; rather, use the implicit layer
        callable behaviour (like ``my_layer(inputs)``), which will apply this method
        with some additional bookkeeping.
        """

        if training is None:
            training = tf.keras.backend.learning_phase()

        states = tf.nest.flatten(states)
        
        # state for the hidden cell
        h = states[:-1]
        # state for the LMU memory
        m = states[-1]
        
        demo_in    = (h[0] - self.thr)/self.thr
        
        
        
        old_spikes = tf.greater(demo_in, 0.)
        old_spikes = tf.cast(old_spikes, dtype=tf.float32)
        
        
        u = inputs
        
        if self.kernel is not None:
            u = tf.matmul(u, self.kernel, name="kernel_matmul")
        
        # separate memory/order dimensions
        m = tf.reshape(m, (-1, self.memory_d, self.order))
        u = tf.expand_dims(u, -1)
       
        A, B = self.A, self.B
        
        _m = tf.matmul(m, A) + tf.matmul(u, B)
        
        m = _m
        
        # re-combine memory/order dimensions
        m = tf.reshape(m, (-1, self.memory_d * self.order))
        
        # apply hidden cell
        
        h_in = m
        
        if hasattr(self.hidden_cell, "state_size"):
            
            o, h = self.hidden_cell(h_in, h, training=training)
        
        else:
            o        = self.hidden_cell(h_in, training=training)
            
            h =      [o]
        
        
        
        #d       = self.decay    
        o_new    = o - self.thr*old_spikes
        
        v_scaled   = (o_new - self.thr) / self.thr
        
        new_z      = SpikeFunction(v_scaled, self.dampening_factor)
        
        return new_z, [o_new] + [m]
    
    
    def get_config(self):
        """Return config of layer (for serialization during model saving/loading)."""

        config = super().get_config()
        config.update(
            dict(
                
                memory_d=self.memory_d,
                order=self.order,
                theta=self._init_theta,
                hidden_cell=tf.keras.layers.serialize(self.hidden_cell),
                kernel_initializer=self.kernel_initializer,
                recurrent_initializer=self.recurrent_initializer,
                kernel_regularizer=self.kernel_regularizer,
                recurrent_regularizer=self.recurrent_regularizer,
                discretizer=self.discretizer,
                bias_initializer=self.bias_initializer,
                bias_regularizer=self.bias_regularizer
                
            )
        )

        return config    
       
        
    @classmethod
    def from_config(cls, config):
        """Load model from serialized config."""

        config["hidden_cell"] = tf.keras.layers.deserialize(config["hidden_cell"])
        return super().from_config(config)
    
###################################################################################
# Most important block for making Spiking LMU units with variable threshold 
###################################################################################

class SLMUcell_thr(DropoutRNNCellMixin, tf.keras.layers.Layer):
    def __init__(self, memory_d, order, 
                 theta, hidden_cell, trainable_theta=False, discretizer="zoh",
                 thr=0.03, tau=20, dampening_factor=0.3, kernel_initializer="glorot_uniform",
                 kernel_regularizer=None,
                recurrent_regularizer=None,
                 recurrent_initializer="orthogonal",bias_initializer="zeros", bias_regularizer = None
                ):
        super().__init__()
        #self.num_neurons      = num_neurons
        #self.state_size       = (num_neurons, num_neurons)
        self.decay            = tf.exp(-1 / tau)

        self.dampening_factor = dampening_factor
        self.thr              = None
        
        self.memory_d         = memory_d
        self.order            = order
        self._init_theta      = theta
        self.hidden_cell      = hidden_cell
        self.discretizer      = discretizer
        self.kernel_initializer = kernel_initializer
        self.recurrent_initializer = recurrent_initializer
        self.kernel            = None
        self.recurrent_kernel  = None
        self.bias              = None
        self.theta_inv         = None
        self.A                 = None
        self.B                 = None
        self.bias_initializer  = bias_initializer
        self.kernel_regularizer= kernel_regularizer
        self.bias_regularizer   = bias_regularizer
    
        if self.hidden_cell is None:
            for conn in ("hidden_to_memory", "input_to_hidden"):
                if getattr(self, conn):
                    raise ValueError(f"{conn} must be False if hidden_cell is None")

            self.hidden_output_size = self.memory_d * self.order
            self.hidden_state_size = []
            

        elif hasattr(self.hidden_cell, "state_size"):
            self.hidden_output_size = self.hidden_cell.output_size
            self.hidden_state_size = self.hidden_cell.state_size
        else:
            # TODO: support layers that don't have the `units` attribute
            self.hidden_output_size = self.hidden_cell.units
            self.hidden_state_size = [self.hidden_cell.units]

        self.state_size = tf.nest.flatten(self.hidden_state_size) + [
            self.memory_d * self.order
        ]
        self.output_size = self.hidden_output_size
        
    
    def theta(self):
        """
        Value of the ``theta`` parameter.
        If ``trainable_theta=True`` this returns the trained value, not the
        initial value passed in to the constructor.
        """
        if self.built:
            return 1 / tf.keras.backend.get_value(self.theta_inv)

        return self._init_theta
        

        
    def _gen_AB(self):
        """Generates A and B matrices."""

        # compute analog A/B matrices
        Q = np.arange(self.order, dtype=np.float64)
        R = (2 * Q + 1)[:, None]
        j, i = np.meshgrid(Q, Q)
        A = np.where(i < j, -1, (-1.0) ** (i - j + 1)) * R
        B = (-1.0) ** Q[:, None] * R
        
        
        self._base_A = tf.constant(A.T, dtype=self.dtype)
        self._base_B = tf.constant(B.T, dtype=self.dtype)

        self.A, self.B = SLMUcell._cont2discrete_zoh(self._base_A/self._init_theta, self._base_B/self._init_theta)
        
        
    def _cont2discrete_zoh(A, B):
        

        # combine A/B and pad to make square matrix
        em_upper = tf.concat([A, B], axis=0)
        em = tf.pad(em_upper, [(0, 0), (0, B.shape[0])])

        # compute matrix exponential
        ms = tf.linalg.expm(em)

        # slice A/B back out of combined matrix
        discrt_A = ms[: A.shape[0], : A.shape[1]]
        discrt_B = ms[A.shape[0] :, : A.shape[1]]

        return discrt_A, discrt_B
    
    
    def build(self, input_shape):
        """
        Builds the cell.
        Notes
        -----
        This method should not be called manually; rather, use the implicit layer
        callable behaviour (like ``my_layer(inputs)``), which will apply this method
        with some additional bookkeeping.
        """

        super().build(input_shape)
        
        enc_d   = input_shape[-1]
        
        if self.kernel_initializer is not None:
            self.kernel = self.add_weight(
                name="kernel",
                shape=(enc_d, self.memory_d),
                initializer=self.kernel_initializer,
                regularizer=self.kernel_regularizer,
            )
        
        elif enc_d != self.memory_d:
            raise ValueError(
                f"For LMUCells with no input kernel, the input dimension ({enc_d})"
                f" must equal `memory_d` ({self.memory_d})."
            )
    
        self.theta_inv = tf.constant(1 / self._init_theta, dtype=self.dtype)
        
        self.recurrent_kernel = None
        
        if self.hidden_cell is not None and not self.hidden_cell.built:
            
            hidden_input_d = self.memory_d * self.order
            
            with tf.name_scope(self.hidden_cell.name):
                self.hidden_cell.build((input_shape[0], hidden_input_d))
                
        self._gen_AB()
        
        self.thr = tf.Variable(tf.random.uniform(shape=(self.hidden_output_size,), minval=0.01, maxval=0.09))
        
        #self.thr = tf.Variable(tf.random.uniform(shape=(1,),minval=0.01, maxval=0.09))
        
        
        
    def call(self, inputs, states, training=None):  # noqa: C901
        """
        Apply this cell to inputs.
        Notes
        -----
        This method should not be called manually; rather, use the implicit layer
        callable behaviour (like ``my_layer(inputs)``), which will apply this method
        with some additional bookkeeping.
        """

        if training is None:
            training = tf.keras.backend.learning_phase()

        states = tf.nest.flatten(states)
        
        # state for the hidden cell
        h = states[:-1]
        # state for the LMU memory
        m = states[-1]
        
        demo_in    = (h[0] - self.thr)/self.thr
        
        
        
        old_spikes = tf.greater(demo_in, 0.)
        old_spikes = tf.cast(old_spikes, dtype=tf.float32)
        
        
        u = inputs
        
        if self.kernel is not None:
            u = tf.matmul(u, self.kernel, name="kernel_matmul")
        
        # separate memory/order dimensions
        m = tf.reshape(m, (-1, self.memory_d, self.order))
        u = tf.expand_dims(u, -1)
       
        A, B = self.A, self.B
        
        _m = tf.matmul(m, A) + tf.matmul(u, B)
        
        m = _m
        
        # re-combine memory/order dimensions
        m = tf.reshape(m, (-1, self.memory_d * self.order))
        
        # apply hidden cell
        
        h_in = m
        
        if hasattr(self.hidden_cell, "state_size"):
            
            o, h = self.hidden_cell(h_in, h, training=training)
        
        else:
            o        = self.hidden_cell(h_in, training=training)
            
            h =      [o]
        
        
        
        #d       = self.decay    
        o_new    = o - self.thr*old_spikes
        
        v_scaled   = (o_new - self.thr) / self.thr
        
        new_z      = SpikeFunction(v_scaled, self.dampening_factor)
        
        return new_z, [o_new] + [m]
    
    
    def get_config(self):
        """Return config of layer (for serialization during model saving/loading)."""

        config = super().get_config()
        config.update(
            dict(
                
                memory_d=self.memory_d,
                order=self.order,
                theta=self._init_theta,
                hidden_cell=tf.keras.layers.serialize(self.hidden_cell),
                kernel_initializer=self.kernel_initializer,
                recurrent_initializer=self.recurrent_initializer,
                kernel_regularizer=self.kernel_regularizer,
                recurrent_regularizer=self.recurrent_regularizer,
                discretizer=self.discretizer,
                bias_initializer=self.bias_initializer,
                bias_regularizer=self.bias_regularizer
                
            )
        )

        return config    
       
        
    @classmethod
    def from_config(cls, config):
        """Load model from serialized config."""

        config["hidden_cell"] = tf.keras.layers.deserialize(config["hidden_cell"])
        return super().from_config(config)
    
    
    
    
###################################################################################
# Most important block for making Spiking LMU units with variable threshold 
###################################################################################

class SLMUcell_thr_V(DropoutRNNCellMixin, tf.keras.layers.Layer):
    def __init__(self, memory_d, order, 
                 theta, hidden_cell, trainable_theta=False, discretizer="zoh",
                 thr=0.03, tau=20, dampening_factor=0.3, kernel_initializer="glorot_uniform",
                 kernel_regularizer=None,
                recurrent_regularizer=None,
                 recurrent_initializer="orthogonal",bias_initializer="zeros", bias_regularizer = None
                ):
        super().__init__()
        #self.num_neurons      = num_neurons
        #self.state_size       = (num_neurons, num_neurons)
        self.decay            = tf.exp(-1 / tau)

        self.dampening_factor = dampening_factor
        self.thr              = None
        
        self.memory_d         = memory_d
        self.order            = order
        self._init_theta      = theta
        self.hidden_cell      = hidden_cell
        self.discretizer      = discretizer
        self.kernel_initializer = kernel_initializer
        self.recurrent_initializer = recurrent_initializer
        self.kernel            = None
        self.recurrent_kernel  = None
        self.bias              = None
        self.theta_inv         = None
        self.A                 = None
        self.B                 = None
        self.bias_initializer  = bias_initializer
        self.kernel_regularizer= kernel_regularizer
        self.bias_regularizer   = bias_regularizer
    
        if self.hidden_cell is None:
            for conn in ("hidden_to_memory", "input_to_hidden"):
                if getattr(self, conn):
                    raise ValueError(f"{conn} must be False if hidden_cell is None")

            self.hidden_output_size = self.memory_d * self.order
            self.hidden_state_size = []
            

        elif hasattr(self.hidden_cell, "state_size"):
            self.hidden_output_size = self.hidden_cell.output_size
            self.hidden_state_size = self.hidden_cell.state_size
        else:
            # TODO: support layers that don't have the `units` attribute
            self.hidden_output_size = self.hidden_cell.units
            self.hidden_state_size = [self.hidden_cell.units]

        self.state_size = tf.nest.flatten(self.hidden_state_size) + [
            self.memory_d * self.order
        ]
        self.output_size = self.hidden_output_size
        
    
    def theta(self):
        """
        Value of the ``theta`` parameter.
        If ``trainable_theta=True`` this returns the trained value, not the
        initial value passed in to the constructor.
        """
        if self.built:
            return 1 / tf.keras.backend.get_value(self.theta_inv)

        return self._init_theta
        

        
    def _gen_AB(self):
        """Generates A and B matrices."""

        # compute analog A/B matrices
        Q = np.arange(self.order, dtype=np.float64)
        R = (2 * Q + 1)[:, None]
        j, i = np.meshgrid(Q, Q)
        A = np.where(i < j, -1, (-1.0) ** (i - j + 1)) * R
        B = (-1.0) ** Q[:, None] * R
        
        
        self._base_A = tf.constant(A.T, dtype=self.dtype)
        self._base_B = tf.constant(B.T, dtype=self.dtype)

        self.A, self.B = SLMUcell._cont2discrete_zoh(self._base_A/self._init_theta, self._base_B/self._init_theta)
        
        
    def _cont2discrete_zoh(A, B):
        

        # combine A/B and pad to make square matrix
        em_upper = tf.concat([A, B], axis=0)
        em = tf.pad(em_upper, [(0, 0), (0, B.shape[0])])

        # compute matrix exponential
        ms = tf.linalg.expm(em)

        # slice A/B back out of combined matrix
        discrt_A = ms[: A.shape[0], : A.shape[1]]
        discrt_B = ms[A.shape[0] :, : A.shape[1]]

        return discrt_A, discrt_B
    
    
    def build(self, input_shape):
        """
        Builds the cell.
        Notes
        -----
        This method should not be called manually; rather, use the implicit layer
        callable behaviour (like ``my_layer(inputs)``), which will apply this method
        with some additional bookkeeping.
        """

        super().build(input_shape)
        
        enc_d   = input_shape[-1]
        
        if self.kernel_initializer is not None:
            self.kernel = self.add_weight(
                name="kernel",
                shape=(enc_d, self.memory_d),
                initializer=self.kernel_initializer,
                regularizer=self.kernel_regularizer,
            )
        
        elif enc_d != self.memory_d:
            raise ValueError(
                f"For LMUCells with no input kernel, the input dimension ({enc_d})"
                f" must equal `memory_d` ({self.memory_d})."
            )
    
        self.theta_inv = tf.constant(1 / self._init_theta, dtype=self.dtype)
        
        self.recurrent_kernel = None
        
        if self.hidden_cell is not None and not self.hidden_cell.built:
            
            hidden_input_d = self.memory_d * self.order
            
            with tf.name_scope(self.hidden_cell.name):
                self.hidden_cell.build((input_shape[0], hidden_input_d))
                
        self._gen_AB()
        
        self.thr = tf.Variable(tf.random.uniform(shape=(self.hidden_output_size,), minval=0.01, maxval=0.09))
        
        #self.thr = tf.Variable(tf.random.uniform(shape=(1,),minval=0.01, maxval=0.09))
        
        
        
    def call(self, inputs, states, training=None):  # noqa: C901
        """
        Apply this cell to inputs.
        Notes
        -----
        This method should not be called manually; rather, use the implicit layer
        callable behaviour (like ``my_layer(inputs)``), which will apply this method
        with some additional bookkeeping.
        """

        if training is None:
            training = tf.keras.backend.learning_phase()

        states = tf.nest.flatten(states)
        
        # state for the hidden cell
        h = states[:-1]
        # state for the LMU memory
        m = states[-1]
        
        demo_in    = (h[0] - self.thr)/self.thr
        
        
        
        old_spikes = tf.greater(demo_in, 0.)
        old_spikes = tf.cast(old_spikes, dtype=tf.float32)
        
        
        u = inputs
        
        if self.kernel is not None:
            u = tf.matmul(u, self.kernel, name="kernel_matmul")
        
        # separate memory/order dimensions
        m = tf.reshape(m, (-1, self.memory_d, self.order))
        u = tf.expand_dims(u, -1)
       
        A, B = self.A, self.B
        
        _m = tf.matmul(m, A) + tf.matmul(u, B)
        
        m = _m
        
        # re-combine memory/order dimensions
        m = tf.reshape(m, (-1, self.memory_d * self.order))
        
        # apply hidden cell
        
        h_in = m
        
        if hasattr(self.hidden_cell, "state_size"):
            
            o, h = self.hidden_cell(h_in, h, training=training)
        
        else:
            o        = self.hidden_cell(h_in, training=training)
            
            h =      [o]
        
        
        
        #d       = self.decay    
        o_new    = o - self.thr*old_spikes
        
        v_scaled   = (o_new - self.thr) / self.thr
        
        new_z      = SpikeFunction(v_scaled, self.dampening_factor)
        
        return tf.math.tanh(o_new), [o_new] + [m]
    
    
    def get_config(self):
        """Return config of layer (for serialization during model saving/loading)."""

        config = super().get_config()
        config.update(
            dict(
                
                memory_d=self.memory_d,
                order=self.order,
                theta=self._init_theta,
                hidden_cell=tf.keras.layers.serialize(self.hidden_cell),
                kernel_initializer=self.kernel_initializer,
                recurrent_initializer=self.recurrent_initializer,
                kernel_regularizer=self.kernel_regularizer,
                recurrent_regularizer=self.recurrent_regularizer,
                discretizer=self.discretizer,
                bias_initializer=self.bias_initializer,
                bias_regularizer=self.bias_regularizer
                
            )
        )

        return config    
       
        
    @classmethod
    def from_config(cls, config):
        """Load model from serialized config."""

        config["hidden_cell"] = tf.keras.layers.deserialize(config["hidden_cell"])
        return super().from_config(config)
    
###################################################################################
# Most important block for making Spiking LMU units with variable threshold 
###################################################################################

class SLMUcell_LIF_thr(DropoutRNNCellMixin, tf.keras.layers.Layer):
    def __init__(self, memory_d, order, 
                 theta, hidden_cell, trainable_theta=False, discretizer="zoh",
                 thr=0.03, tau=20, dampening_factor=0.3, kernel_initializer="glorot_uniform",
                 kernel_regularizer=None,
                recurrent_regularizer=None,
                 recurrent_initializer="orthogonal",bias_initializer="zeros", bias_regularizer = None
                ):
        super().__init__()
        #self.num_neurons      = num_neurons
        #self.state_size       = (num_neurons, num_neurons)
        self.decay            = tf.exp(-1 / tau)

        self.dampening_factor = dampening_factor
        self.thr              = None
        
        self.memory_d         = memory_d
        self.order            = order
        self._init_theta      = theta
        self.hidden_cell      = hidden_cell
        self.discretizer      = discretizer
        self.kernel_initializer = kernel_initializer
        self.recurrent_initializer = recurrent_initializer
        self.kernel            = None
        self.recurrent_kernel  = None
        self.bias              = None
        self.theta_inv         = None
        self.A                 = None
        self.B                 = None
        self.bias_initializer  = bias_initializer
        self.kernel_regularizer= kernel_regularizer
        self.bias_regularizer   = bias_regularizer
    
        if self.hidden_cell is None:
            for conn in ("hidden_to_memory", "input_to_hidden"):
                if getattr(self, conn):
                    raise ValueError(f"{conn} must be False if hidden_cell is None")

            self.hidden_output_size = self.memory_d * self.order
            self.hidden_state_size = []
            

        elif hasattr(self.hidden_cell, "state_size"):
            self.hidden_output_size = self.hidden_cell.output_size
            self.hidden_state_size = self.hidden_cell.state_size
        else:
            # TODO: support layers that don't have the `units` attribute
            self.hidden_output_size = self.hidden_cell.units
            self.hidden_state_size = [self.hidden_cell.units]

        self.state_size = tf.nest.flatten(self.hidden_state_size) + [
            self.memory_d * self.order
        ]
        self.output_size = self.hidden_output_size
        
    
    def theta(self):
        """
        Value of the ``theta`` parameter.
        If ``trainable_theta=True`` this returns the trained value, not the
        initial value passed in to the constructor.
        """
        if self.built:
            return 1 / tf.keras.backend.get_value(self.theta_inv)

        return self._init_theta
        

        
    def _gen_AB(self):
        """Generates A and B matrices."""

        # compute analog A/B matrices
        Q = np.arange(self.order, dtype=np.float64)
        R = (2 * Q + 1)[:, None]
        j, i = np.meshgrid(Q, Q)
        A = np.where(i < j, -1, (-1.0) ** (i - j + 1)) * R
        B = (-1.0) ** Q[:, None] * R
        
        
        self._base_A = tf.constant(A.T, dtype=self.dtype)
        self._base_B = tf.constant(B.T, dtype=self.dtype)

        self.A, self.B = SLMUcell._cont2discrete_zoh(self._base_A/self._init_theta, self._base_B/self._init_theta)
        
        
    def _cont2discrete_zoh(A, B):
        

        # combine A/B and pad to make square matrix
        em_upper = tf.concat([A, B], axis=0)
        em = tf.pad(em_upper, [(0, 0), (0, B.shape[0])])

        # compute matrix exponential
        ms = tf.linalg.expm(em)

        # slice A/B back out of combined matrix
        discrt_A = ms[: A.shape[0], : A.shape[1]]
        discrt_B = ms[A.shape[0] :, : A.shape[1]]

        return discrt_A, discrt_B
    
    
    def build(self, input_shape):
        """
        Builds the cell.
        Notes
        -----
        This method should not be called manually; rather, use the implicit layer
        callable behaviour (like ``my_layer(inputs)``), which will apply this method
        with some additional bookkeeping.
        """

        super().build(input_shape)
        
        enc_d   = input_shape[-1]
        
        if self.kernel_initializer is not None:
            self.kernel = self.add_weight(
                name="kernel",
                shape=(enc_d, self.memory_d),
                initializer=self.kernel_initializer,
                regularizer=self.kernel_regularizer,
            )
        
        elif enc_d != self.memory_d:
            raise ValueError(
                f"For LMUCells with no input kernel, the input dimension ({enc_d})"
                f" must equal `memory_d` ({self.memory_d})."
            )
    
        self.theta_inv = tf.constant(1 / self._init_theta, dtype=self.dtype)
        
        self.recurrent_kernel = None
        
        if self.hidden_cell is not None and not self.hidden_cell.built:
            
            hidden_input_d = self.memory_d * self.order
            
            with tf.name_scope(self.hidden_cell.name):
                self.hidden_cell.build((input_shape[0], hidden_input_d))
                
        self._gen_AB()
        
        self.thr = tf.Variable(tf.random.uniform(shape=(self.hidden_output_size,),minval=0.01, maxval=0.09))
        
        #self.thr = tf.Variable(tf.random.uniform(shape=(1,),minval=0.01, maxval=0.09))
        
        
        
    def call(self, inputs, states, training=None):  # noqa: C901
        """
        Apply this cell to inputs.
        Notes
        -----
        This method should not be called manually; rather, use the implicit layer
        callable behaviour (like ``my_layer(inputs)``), which will apply this method
        with some additional bookkeeping.
        """

        if training is None:
            training = tf.keras.backend.learning_phase()

        states = tf.nest.flatten(states)
        
        # state for the hidden cell
        h = states[:-1]
        # state for the LMU memory
        m = states[-1]
        
        demo_in    = (h[0] - self.thr)/self.thr
        
        
        
        old_spikes = tf.greater(demo_in, 0.)
        old_spikes = tf.cast(old_spikes, dtype=tf.float32)
        
        
        u = inputs
        
        if self.kernel is not None:
            u = tf.matmul(u, self.kernel, name="kernel_matmul")
        
        # separate memory/order dimensions
        m = tf.reshape(m, (-1, self.memory_d, self.order))
        u = tf.expand_dims(u, -1)
       
        A, B = self.A, self.B
        
        _m = tf.matmul(m, A) + tf.matmul(u, B)
        
        m = _m
        
        # re-combine memory/order dimensions
        m = tf.reshape(m, (-1, self.memory_d * self.order))
        
        # apply hidden cell
        
        h_in = m
        
        if hasattr(self.hidden_cell, "state_size"):
            
            o, h = self.hidden_cell(h_in, h, training=training)
        
        else:
            o        = self.hidden_cell(h_in, training=training)
            
            h        = [o]
        
        
        
        d        = self.decay    
        o_new    =  o - self.thr*old_spikes
        
        v_scaled   = (o_new - self.thr) / self.thr
        
        new_z      = SpikeFunction(v_scaled, self.dampening_factor)
        
        return new_z*tf.math.tanh(o_new), [o_new] + [m]
    
    
    def get_config(self):
        """Return config of layer (for serialization during model saving/loading)."""

        config = super().get_config()
        config.update(
            dict(
                
                memory_d=self.memory_d,
                order=self.order,
                theta=self._init_theta,
                hidden_cell=tf.keras.layers.serialize(self.hidden_cell),
                kernel_initializer=self.kernel_initializer,
                recurrent_initializer=self.recurrent_initializer,
                kernel_regularizer=self.kernel_regularizer,
                recurrent_regularizer=self.recurrent_regularizer,
                discretizer=self.discretizer,
                bias_initializer=self.bias_initializer,
                bias_regularizer=self.bias_regularizer
                
            )
        )

        return config    
       
        
    @classmethod
    def from_config(cls, config):
        """Load model from serialized config."""

        config["hidden_cell"] = tf.keras.layers.deserialize(config["hidden_cell"])
        return super().from_config(config)
    
    
# Decoding layer

class DSNN(tf.keras.layers.Layer):
    def __init__(self, num_neurons, thr=0.03, tau=20., dampening_factor=0.3):
        super().__init__()
        
        self.num_neurons = num_neurons
        self.state_size = (num_neurons, num_neurons)
        self.decay = tf.exp(-1 / tau)

        self.dampening_factor = dampening_factor
        self.thr = thr

        self.input_weights = None
        self.recurrent_weights = None
        self.disconnect_mask = None
        #self.dec_weights     = None
        #self.rec_dec_weights     = None

    def build(self, input_shape):
        n_in = input_shape[-1]
        n = self.num_neurons
        
        rand_init = tf.keras.initializers.RandomNormal

        # define the input weight variable
        self.input_weights = self.add_weight(
            shape=(n_in,n),
            initializer=rand_init(stddev=1. / np.sqrt(n_in)),
            name='input_weights')
        
        # define the recurrent weight variable
        self.disconnect_mask = tf.cast(np.diag(np.ones(n, dtype=bool)), tf.bool)
        self.recurrent_weights = self.add_weight(
            shape=(n, n),
            initializer=rand_init(stddev=1. / np.sqrt(n)),
            name='recurrent_weights')
        #self.thr = tf.Variable(tf.random.uniform(shape=(self.num_neurons,),minval=0.01, maxval=0.09)[0])
#         self.dec_weights      = self.add_weight(
#             shape=(n, n),
#             initializer=rand_init(stddev=1. / np.sqrt(n)),
#             name='dec_weights')
        
#         self.rec_dec_weights      = self.add_weight(
#             shape=(n, n),
#             initializer=rand_init(stddev=1. / np.sqrt(n)),
#             name='rec_dec_weights')

        super().build(input_shape)

    def get_recurrent_weights(self):
        
        w_rec = tf.where(self.disconnect_mask, tf.zeros_like(self.recurrent_weights), self.recurrent_weights)
        
        return w_rec

    def call(self, inputs, states, constants=None):
        
        old_v     = states[0]
        old_spike = states[1]

        # compute the input currents
        w_rec = self.get_recurrent_weights()
        i_in = inputs @ self.input_weights + old_v @ w_rec
      
        # update the voltage
        d = self.decay
        i_reset = - self.thr * 1
        new_v = (d * old_v + i_in*(1-d)) +i_reset

        v_scaled   = (new_v - self.thr) / self.thr
        new_z      = SpikeFunction(v_scaled, self.dampening_factor)
        
        #dec_output = new_v @ self.dec_weights + new_z @ self.rec_dec_weights
        
        new_state = (new_v, new_z)
        return new_v, new_state

In [6]:
class RSNN(tf.keras.layers.Layer):
    def __init__(self, num_neurons, thr=0.03, tau=20., dampening_factor=0.3):
        super().__init__()
        
        self.num_neurons = num_neurons
        self.state_size = (num_neurons, num_neurons)
        self.decay = tf.exp(-1 / tau)

        self.dampening_factor = dampening_factor
        self.thr = thr

        self.input_weights = None
        self.recurrent_weights = None
        self.disconnect_mask = None


    def build(self, input_shape):
        n_in = input_shape[-1]
        n    = self.num_neurons
        
        rand_init = tf.keras.initializers.RandomNormal

        # define the input weight variable
        self.input_weights = self.add_weight(
            shape=(n_in,n),
            initializer=rand_init(stddev=1. / np.sqrt(n_in)),
            name='input_weights')
        
        # define the recurrent weight variable
        self.disconnect_mask = tf.cast(np.diag(np.ones(n, dtype=bool)), tf.bool)
        self.recurrent_weights = self.add_weight(
            shape=(n, n),
            initializer=rand_init(stddev=1. / np.sqrt(n)),
            name='recurrent_weights')
        


        super().build(input_shape)

    def get_recurrent_weights(self):
        
        w_rec = tf.where(self.disconnect_mask, tf.zeros_like(self.recurrent_weights), self.recurrent_weights)
        
        return w_rec

    def call(self, inputs, states, constants=None):
        
        old_v     = states[0]
        old_spike = states[1]

        # compute the input currents
        w_rec = self.get_recurrent_weights()
        i_in = inputs @ self.input_weights + old_spike @ w_rec
      
        # update the voltage
        d = self.decay
        i_reset = - self.thr * old_spike
        new_v = d * old_v + (1-d) * i_in + i_reset

        v_scaled   = (new_v - self.thr) / self.thr
        new_z      = SpikeFunction(v_scaled, self.dampening_factor)
        
        
        new_state = (new_v, new_z)
        return new_z*new_v,new_state

In [7]:
class RSNN_vthr(tf.keras.layers.Layer):
    def __init__(self, num_neurons, tau=20., dampening_factor=0.3):
        super().__init__()
        
        self.num_neurons = num_neurons
        self.state_size = (num_neurons, num_neurons)
        self.decay = tf.exp(-1 / tau)

        self.dampening_factor = dampening_factor
        #self.thr = thr

        self.input_weights = None
        self.recurrent_weights = None
        self.disconnect_mask = None
        self.thr=None

    def build(self, input_shape):
        n_in = input_shape[-1]
        n    = self.num_neurons
        
        rand_init = tf.keras.initializers.RandomNormal

        # define the input weight variable
        self.input_weights = self.add_weight(
            shape=(n_in,n),
            initializer=rand_init(stddev=1. / np.sqrt(n_in)),
            name='input_weights')
        
        # define the recurrent weight variable
        self.disconnect_mask = tf.cast(np.diag(np.ones(n, dtype=bool)), tf.bool)
        self.recurrent_weights = self.add_weight(
            shape=(n, n),
            initializer=rand_init(stddev=1. / np.sqrt(n)),
            name='recurrent_weights')
        
        self.thr = tf.Variable(tf.random.uniform(shape=(1,),minval=-1, maxval=1)[0])

        super().build(input_shape)

    def get_recurrent_weights(self):
        
        w_rec = tf.where(self.disconnect_mask, tf.zeros_like(self.recurrent_weights), self.recurrent_weights)
        
        return w_rec
    
    def get_threshold(self):
        
        return self.thr

    def call(self, inputs, states, constants=None):
        
        old_v     = states[0]
        old_spike = states[1]

        # compute the input currents
        w_rec = self.get_recurrent_weights()
        i_in = inputs @ self.input_weights + old_spike @ w_rec
      
        # update the voltage
        d = self.decay
        i_reset = - self.thr * old_spike
        new_v = d * old_v + (1-d) * i_in + i_reset

        v_scaled   = (new_v - self.thr) / self.thr
        new_z      = SpikeFunction(v_scaled, self.dampening_factor)
        
        
        new_state = (new_v, new_z)
        return tf.math.tanh(new_z*new_v),new_state
    
    def get_config(self):
        
        config = super().get_config()
        
        config.update({
            'thr': self.thr,
        })
        return config

In [8]:
def my_loss(y_true,y_pred):
    
    squared_difference = tf.square(y_true - y_pred)
    
    return tf.reduce_mean(squared_difference, axis=[1, 0])

In [9]:
def loss_function(pred_y, y):
    return keras_backend.mean(keras.losses.mean_squared_error(y, pred_y))

# def my_loss(y_true,y_pred):
    
#     squared_difference = tf.square(y_true - y_pred)
    
#     return tf.reduce_sum(tf.reduce_sum(squared_difference, axis=1)/(y_pred.shape[1]),axis=0,)/(y_pred.shape[0])

def my_loss_(y_true,y_pred):
    
    squared_difference = tf.square(y_true - y_pred)
    
    return tf.reduce_sum(tf.reduce_sum(squared_difference, axis=1)/(y_true.shape[1]),axis=0,)/(62)



def compute_loss(model, x, y, loss_fn=my_loss):
    logits = model.forward(x)
    mse = loss_fn(y, logits)
    return mse, logits

In [10]:
class TrainModel(keras.Model):
    def __init__(self):
        super().__init__()
        
        #self.hidden1 = keras.layers.LSTM(128, input_shape=(None,6,), activation='tanh', return_sequences=True,unroll=True)
        #self.hidden2 = keras.layers.LSTM(128, activation='tanh', return_sequences=True,unroll=True)
        #self.hidden3 = keras.layers.LSTM(128, activation='tanh', return_sequences=True,unroll=True)
        #self.hidden4 = keras.layers.LSTM(48, activation='tanh', return_sequences=True,unroll=True)
        #self.hidden5 = keras.layers.LSTM(208, activation='tanh', return_sequences=True,unroll=True)
        #self.out     = keras.layers.LSTM(1,activation='tanh', return_sequences=True,unroll=True)
                 
        cell1                      = RSNN(208)
        cell2                      = RSNN(208)
        cell3                      = RSNN(208)

        self.hidden1                      = tf.keras.layers.RNN(cell1, return_sequences=True, unroll=True)
        self.hidden2                      = tf.keras.layers.RNN(cell2, return_sequences=True, unroll=True)
        self.hidden3                      = tf.keras.layers.RNN(cell3, return_sequences=True, unroll=True)        
        
        Dec                               = DSNN(208)

        self.Deco                          = tf.keras.layers.RNN(Dec, return_sequences=True)

        
#         cell =  SLMUcell_LIF_thr(memory_d=10,
#             order=10,
#             theta=784,
#             hidden_cell=tf.keras.layers.SimpleRNNCell(units=208))

#         self.LMU                          = tf.keras.layers.RNN(cell, return_sequences=True)

#         cell1                        = SLMUcell_LIF_thr(memory_d=10,
#             order=10,
#             theta=784,
#             hidden_cell=tf.keras.layers.SimpleRNNCell(units=208))

#         self.LMU1                         = tf.keras.layers.RNN(cell1, return_sequences=True)

# #         cell2 =  SLMUcell_LIF_thr(memory_d=10,
# #             order=10,
# #             theta=784,
# #             hidden_cell=tf.keras.layers.SimpleRNNCell(units=208))

# #         self.LMU2                          = tf.keras.layers.RNN(cell2, return_sequences=True)    
        
        
#         cell3 =  SLMUcell_thr_V(memory_d=10,
#         order=10,
#         theta=784,
#         hidden_cell=tf.keras.layers.SimpleRNNCell(units=208))

#         self.LMU3                          = tf.keras.layers.RNN(cell3, return_sequences=True)

        self.d1                            = Dense(240, activation='tanh')
        self.d2                            = Dense(64, activation='tanh')
        self.d3                            = Dense(1, activation='tanh')
        
    
        
    def forward(self, x):
        
        x = self.hidden1(x)

        x = self.hidden2(x)
        
        #x = self.LMU2(x)
        
        x = self.hidden3(x)
        
        x = self.Deco(x)

        x = self.d1(x)
        
        x = self.d2(x)
        
        x = self.d3(x)
        
        
        return x
    

def copy_model(model, x):
    '''Copy model weights to a new model.
    
    Args:
        model: model to be copied.
        x: An input example. This is used to run
            a forward pass in order to add the weights of the graph
            as variables.
    Returns:
        A copy of the model.
    '''
    copied_model = TrainModel()
    
    # If we don't run this step the weights are not "initialized"
    # and the gradients will not be computed.
    copied_model.forward(tf.convert_to_tensor(x))
    
    copied_model.set_weights(model.get_weights())
    
    return copied_model

In [11]:
input_nn_train  = tf.cast(tf.convert_to_tensor(input_nn_train_inp), dtype=tf.float32) 
output_nn_train = tf.cast(tf.convert_to_tensor(input_nn_train_output), dtype=tf.float32)

input_nn_test  = tf.cast(tf.convert_to_tensor(input_nn_test_inp), dtype=tf.float32) 
output_nn_test = tf.cast(tf.convert_to_tensor(input_nn_test_output), dtype=tf.float32)

In [12]:
def model_builder(hp):
    
    
    model_ = Sequential()
    
    model_.add(Input(shape=(None, 6)))
    
    
    for j in range(hp.Int('SRNN_layers', 1,12,default=1)):
    
        hp_units1 = hp.Int('SRNN_units', min_value=16, max_value=256, step=32)
#         hp_units2 = hp.Int('Memory', min_value=10, max_value=256, step=32)
        
#         cell                      = RSNN_vthr(hp_units1)
        
        cell                      = SLMUcell_LIF_thr(memory_d=10,
            order=10,
            theta=784,
            hidden_cell=tf.keras.layers.SimpleRNNCell(units=hp_units1))
        
        model_.add((tf.keras.layers.RNN(cell, return_sequences=True)))
        #model_.add(Activation('tanh'))
        
    
    #model_.add(LSTM(1, activation='tanh', return_sequences=True))
    #model.add(tf.reduce_mean(axis=2, keepdims=True))
    
    #Dec                           = DSNN(1)
    #model_.add(tf.keras.layers.RNN(Dec, return_sequences=True))
    #model_.add(tf.keras.activations.tanh())
    
    #hp_units3 = hp.Int('Deco_units', min_value=16, max_value=256, step=32)
    
    cell3 =  SLMUcell_thr_V(memory_d=10,
    order=10,
    theta=784,
    hidden_cell=tf.keras.layers.SimpleRNNCell(units=hp_units1))

    #cell1                     = RSNN_vthr(6)

    model_.add(tf.keras.layers.RNN(cell3, return_sequences=True))
    
    #model_.add(Activation('tanh'))



    for j in range(hp.Int('Dense_layers', 1,4,default=1)):

        hp_units2 = hp.Int('units_dense', min_value=16, max_value=256, step=32)

        model_.add(Dense(hp_units2, activation='tanh'))
        
        
    model_.add(Dense(6, activation='tanh'))
    
    
    opt = Adam(0.001)
    
    model_.compile(optimizer = opt, loss = my_loss)
    
    
    
    return model_

In [ ]:
tuner = kt.Hyperband(model_builder,
                     objective='loss',
                     max_epochs=1000,
                     factor=3,
                     directory='my_dir',
                     project_name='intro_to_kt_30_31', hyperband_iterations=5)

In [ ]:
tuner.search(input_nn_train_inp, input_nn_train_output, batch_size = 32, epochs=1000)

In [ ]:
best_hp = tuner.get_best_hyperparameters()[0]
model_  = tuner.hypermodel.build(best_hp)

In [ ]:
model_.summary()

In [13]:
# cell =  SLMUcell_LIF_thr(memory_d=10,
#     order=10,
#     theta=784,
#     hidden_cell=tf.keras.layers.SimpleRNNCell(units=208))

# LMU                          = tf.keras.layers.RNN(cell, return_sequences=True)

# cell1                        = SLMUcell_LIF_thr(memory_d=10,
#     order=10,
#     theta=784,
#     hidden_cell=tf.keras.layers.SimpleRNNCell(units=208))

# LMU1                         = tf.keras.layers.RNN(cell1, return_sequences=True)

# cell2 =  SLMUcell_LIF_thr(memory_d=10,
#     order=10,
#     theta=784,
#     hidden_cell=tf.keras.layers.SimpleRNNCell(units=208))

# LMU2                          = tf.keras.layers.RNN(cell2, return_sequences=True)


# cell3 =  SLMUcell_thr_V(memory_d=10,
#     order=10,
#     theta=784,
#     hidden_cell=tf.keras.layers.SimpleRNNCell(units=208))

# LMU3                          = tf.keras.layers.RNN(cell3, return_sequences=True)

# cell3                        = SLMUcell_LIF_thr(memory_d=10,
#     order=10,
#     theta=784,
#     hidden_cell=tf.keras.layers.SimpleRNNCell(units=128))

#LMU3                         = tf.keras.layers.RNN(cell1, return_sequences=True, return_state=True)
#decoder                      = tf.keras.layers.RNN(cell3, return_sequences=True)


cell1                           = RSNN(208)
cell2                           = RSNN(208)
cell3                           = RSNN(208)

cell4                           = DSNN(208)
# cell4                           = RSNN_vthr(112)
# cell5                           = RSNN_vthr(240)
# cell6                           = RSNN_vthr(240)
# cell7                           = RSNN_vthr(240)
# cell8                           = RSNN_vthr(240)
# cell9                           = RSNN_vthr(240)
# cell10                           = RSNN_vthr(240)
# cell11                           = RSNN_vthr(240)
# cell12                           = RSNN_vthr(240)


layer1                          = tf.keras.layers.RNN(cell1, return_sequences=True)

layer2                          = tf.keras.layers.RNN(cell2, return_sequences=True)

layer3                          = tf.keras.layers.RNN(cell3, return_sequences=True)

layer4                          = tf.keras.layers.RNN(cell4, return_sequences=True)

# layer5                          = tf.keras.layers.RNN(cell5, return_sequences=True)

# layer6                          = tf.keras.layers.RNN(cell6, return_sequences=True)

# layer7                          = tf.keras.layers.RNN(cell7, return_sequences=True)

# layer8                          = tf.keras.layers.RNN(cell8, return_sequences=True)

# layer9                          = tf.keras.layers.RNN(cell9, return_sequences=True)

# layer10                          = tf.keras.layers.RNN(cell10, return_sequences=True)

# layer11                          = tf.keras.layers.RNN(cell11, return_sequences=True)

# layer12                          = tf.keras.layers.RNN(cell12, return_sequences=True)




# # Dec1                         = keras_lmu.LMU(10, 10, 784, tf.keras.layers.SimpleRNNCell(units=64),return_sequences=True)
# # Dec2                         = keras_lmu.LMU(10, 10, 784, tf.keras.layers.SimpleRNNCell(units=64),return_sequences=True)
# # Dec3                         = keras_lmu.LMU(10, 10, 784, tf.keras.layers.SimpleRNNCell(units=64),return_sequences=True)

inputs = Input((None, 4))
# # h      = Input((64,))
# # c      = Input((100,))
# #X   = Dec1(inputs)
# #X   = tf.tanh(X)
#X   = LSTM(128, return_sequences=True)(inputs)
#X   = LSTM(128, return_sequences=True)(X)
#X   = LMU2(X)
#V   = LSTM(128, return_sequences=True)(X)

# #X1 = tf.reduce_mean((X1),axis=2, keepdims=True)
# # o = Dense(64, activation='tanh')(X1)
# # o = Dense(64, activation='tanh')(o)
# # o = Dense(1, activation='tanh')(o)
# # X1 = Dense(64, activation='tanh')(X1)
# # X1   = Dec1(S)
# # X1   = tf.tanh(X1)
# # X1   = Dec2(X1)
# # X1   = tf.tanh(X1)
# # o    = Dec3(X1)
# # o   = tf.tanh(o)
X    = layer1(inputs)
X    = layer2(X)
X    = layer3(X)
V    = layer4(X)

# X  = layer1(inputs)
# X  = layer2(X)
# X  = layer3(X)
# X  = layer4(X)
#X  = layer5(X)
# X  = layer6(X)
# X  = layer7(X)
# X  = layer8(X)
# X  = layer9(X)
# X  = layer10(X)
# X  = layer11(X)
# X  = layer12(X)
V  = Dense(240, activation='tanh')(V)
V  = Dense(64, activation='tanh')(V)
V1  = Dense(1, activation='tanh')(V)




model2 = Model(inputs=inputs, outputs=[V1])


opt = Adam(0.001)

model2.compile(optimizer = opt, loss = my_loss)

In [14]:
cell11                           = RSNN(208)
cell21                           = RSNN(208)
cell31                           = RSNN(208)

cell41                           = DSNN(208)
# cell4                           = RSNN_vthr(112)
# cell5                           = RSNN_vthr(240)
# cell6                           = RSNN_vthr(240)
# cell7                           = RSNN_vthr(240)
# cell8                           = RSNN_vthr(240)
# cell9                           = RSNN_vthr(240)
# cell10                           = RSNN_vthr(240)
# cell11                           = RSNN_vthr(240)
# cell12                           = RSNN_vthr(240)


layer11                          = tf.keras.layers.RNN(cell11, return_sequences=True)

layer21                          = tf.keras.layers.RNN(cell21, return_sequences=True)

layer31                          = tf.keras.layers.RNN(cell31, return_sequences=True)

layer41                          = tf.keras.layers.RNN(cell41, return_sequences=True)

# layer5                          = tf.keras.layers.RNN(cell5, return_sequences=True)

# layer6                          = tf.keras.layers.RNN(cell6, return_sequences=True)

# layer7                          = tf.keras.layers.RNN(cell7, return_sequences=True)

# layer8                          = tf.keras.layers.RNN(cell8, return_sequences=True)

# layer9                          = tf.keras.layers.RNN(cell9, return_sequences=True)

# layer10                          = tf.keras.layers.RNN(cell10, return_sequences=True)

# layer11                          = tf.keras.layers.RNN(cell11, return_sequences=True)

# layer12                          = tf.keras.layers.RNN(cell12, return_sequences=True)




# # Dec1                         = keras_lmu.LMU(10, 10, 784, tf.keras.layers.SimpleRNNCell(units=64),return_sequences=True)
# # Dec2                         = keras_lmu.LMU(10, 10, 784, tf.keras.layers.SimpleRNNCell(units=64),return_sequences=True)
# # Dec3                         = keras_lmu.LMU(10, 10, 784, tf.keras.layers.SimpleRNNCell(units=64),return_sequences=True)

inputs1 = Input((None, 4))
# # h      = Input((64,))
# # c      = Input((100,))
# #X   = Dec1(inputs)
# #X   = tf.tanh(X)
#X   = LSTM(128, return_sequences=True)(inputs)
#X   = LSTM(128, return_sequences=True)(X)
#X   = LMU2(X)
#V   = LSTM(128, return_sequences=True)(X)

# #X1 = tf.reduce_mean((X1),axis=2, keepdims=True)
# # o = Dense(64, activation='tanh')(X1)
# # o = Dense(64, activation='tanh')(o)
# # o = Dense(1, activation='tanh')(o)
# # X1 = Dense(64, activation='tanh')(X1)
# # X1   = Dec1(S)
# # X1   = tf.tanh(X1)
# # X1   = Dec2(X1)
# # X1   = tf.tanh(X1)
# # o    = Dec3(X1)
# # o   = tf.tanh(o)
X    = layer11(inputs1)
X    = layer21(X)
X    = layer31(X)
V    = layer41(X)

# X  = layer1(inputs)
# X  = layer2(X)
# X  = layer3(X)
# X  = layer4(X)
#X  = layer5(X)
# X  = layer6(X)
# X  = layer7(X)
# X  = layer8(X)
# X  = layer9(X)
# X  = layer10(X)
# X  = layer11(X)
# X  = layer12(X)
V  = Dense(240, activation='tanh')(V)
V  = Dense(64, activation='tanh')(V)
V1  = Dense(1, activation='tanh')(V)




model3 = Model(inputs=inputs1, outputs=[V1])


opt__ = Adam(0.001)

model3.compile(optimizer = opt__, loss = my_loss)

In [15]:
# Defing Early stopping accuracy to avoid overfitting

class EarlyStoppingByAccuracy(Callback): 
    
    def __init__(self, monitor='val_accuracy', value=0.95, verbose=0):
        super(Callback, self).__init__()
        self.monitor = monitor
        self.value = value
        self.verbose = verbose

    def on_epoch_end(self, epoch, logs={}):
        current = logs.get(self.monitor)
        if current is None:
            warnings.warn("Early stopping requires %s available!" % self.monitor, RuntimeWarning)

        if current <= self.value:
            if self.verbose > 0:
                print("Epoch %05d: early stopping THR" % epoch)
            self.model.stop_training = True

In [18]:
callbacks = [
                 EarlyStoppingByAccuracy(monitor='loss', value=1e-3, verbose=1)
            ]

In [19]:
model2.fit(input_nn_train_inp, input_nn_train_output, epochs=1000, batch_size=32, callbacks=callbacks)

Epoch 1/1000
1/1 [==============================] - 1s 1s/step - loss: 0.0013
Epoch 2/1000
1/1 [==============================] - 1s 1s/step - loss: 0.0012
Epoch 3/1000
1/1 [==============================] - 1s 1s/step - loss: 0.0014
Epoch 4/1000
1/1 [==============================] - 1s 1s/step - loss: 0.0014
Epoch 5/1000
1/1 [==============================] - 1s 1s/step - loss: 0.0014
Epoch 6/1000
1/1 [==============================] - 1s 1s/step - loss: 0.0012
Epoch 7/1000
1/1 [==============================] - 1s 1s/step - loss: 0.0011
Epoch 8/1000
1/1 [==============================] - 1s 1s/step - loss: 0.0010
Epoch 9/1000
1/1 [==============================] - 1s 1s/step - loss: 0.0010
Epoch 10/1000
1/1 [==============================] - 1s 1s/step - loss: 9.4049e-04


In [ ]:
model2.summary()

In [20]:
batch_size    = 32

lr_inner      = 0.001
model         = TrainModel()
optimizer     = keras.optimizers.Adam(0.001)


train_dataset = tf.data.Dataset.from_tensor_slices((input_nn_train[:,:,:], output_nn_train[:,:,:]))
train_dataset = train_dataset.shuffle(10000).batch(batch_size)

In [21]:
random_   =  model.forward(input_nn_train)

In [23]:
def MAML(model, train_dataset, input_nn_test, output_nn_test, epochs):

    lr_inner = 0.002
    for _ in range(epochs):

        total_loss = 0
        losses = []
        #start = time.time()
        # Step 3 and 4
        for i, (x, y) in enumerate(train_dataset):
        #for i in range(0, len(input_nn_train), batch_size):

            #x = input_nn_train[i:i + batch_size, :]
            #y = output_nn_train[i:i + batch_size, :]
            #x, y = np_to_tensor(t.batch())
            model.forward(x)  # run forward pass to initialize weights

            with tf.GradientTape() as test_tape:
                # test_tape.watch(model.trainable_variables)
                # Step 5
                with tf.GradientTape() as train_tape:
                    train_loss, _ = compute_loss(model, x, y)

                # Step 6
                gradients = train_tape.gradient(train_loss, model.trainable_variables)
                k = 0
                
                model_copy = copy_model(model, x)

                for j in range(len(model_copy.layers)):

                    model_copy.layers[j].cell.kernel = tf.subtract(model.layers[j].cell.kernel,
                                tf.multiply(lr_inner, gradients[k]))
                    model_copy.layers[j].cell.recurrent_kernel = tf.subtract(model.layers[j].cell.recurrent_kernel,
                            tf.multiply(lr_inner, gradients[k+1]))
                    model_copy.layers[j].cell.bias = tf.subtract(model.layers[j].cell.bias,
                                tf.multiply(lr_inner, gradients[k+2]))
                    k += 3
                    

                # Step 8
                test_loss, logits = compute_loss(model_copy, input_nn_test, output_nn_test)

            # Step 8
            gradients_ = test_tape.gradient(test_loss, model.trainable_variables)
            optimizer.apply_gradients(zip(gradients_, model.trainable_variables))

            # Logs
            total_loss += test_loss
            loss = total_loss / (i+1.0)
            losses.append(loss)

    #         if loss.numpy()<1e-2:
    #             break
        print(loss)

        if loss.numpy()<1e-4:
            break
                #start = time.time()
                
    return loss


def MAML_RLIF(model, train_dataset, input_nn_test, output_nn_test, epochs):

    for tt in range(epochs):

        total_loss = 0
        losses = []
        #start = time.time()
        # Step 3 and 4
        for i, (x, y) in enumerate(train_dataset):
        #for i in range(0, len(input_nn_train), batch_size):

            #x = input_nn_train[i:i + batch_size, :]
            #y = output_nn_train[i:i + batch_size, :]
            #x, y = np_to_tensor(t.batch())
            model.forward(x)  # run forward pass to initialize weights

            with tf.GradientTape() as test_tape:
                # test_tape.watch(model.trainable_variables)
                # Step 5
                with tf.GradientTape() as train_tape:
                    train_loss, _ = compute_loss(model, x, y)

                # Step 6
                gradients = train_tape.gradient(train_loss, model.trainable_variables)
                k = 0
                
                model_copy = copy_model(model, x)

                for j in range(len(model_copy.layers)):

                    model_copy.layers[j].cell.input_weights = tf.subtract(model.layers[j].cell.input_weights,
                                tf.multiply(lr_inner, gradients[k]))
                    model_copy.layers[j].cell.recurrent_weights = tf.subtract(model.layers[j].cell.recurrent_weights,
                            tf.multiply(lr_inner, gradients[k+1]))
                    model_copy.layers[j].cell.thr = tf.subtract(model.layers[j].cell.thr,
                                tf.multiply(lr_inner, gradients[k+2]))
                    k += 3
                    

                # Step 8
                test_loss, logits = compute_loss(model_copy, input_nn_test, output_nn_test)

            # Step 8
            gradients_ = test_tape.gradient(test_loss, model.trainable_variables)
            optimizer.apply_gradients(zip(gradients_, model.trainable_variables))

            # Logs
            total_loss += test_loss
            loss = total_loss / (i+1.0)
            losses.append(loss)

    #         if loss.numpy()<1e-2:
    #             break
        print(loss)

        if loss.numpy()<1e-4:
            break
                #start = time.time()
                
    return loss




def MAML_LMU(model, train_dataset, input_nn_test, output_nn_test, epochs):

    lr_inner = 0.001
    for tt in range(epochs):

        total_loss = 0
        losses = []
        #start = time.time()
        meta_grads = [tf.zeros_like(w) for w in model.trainable_variables]
        # Step 3 and 4
        for i, (x, y) in enumerate(train_dataset):
        #for i in range(0, len(input_nn_train), batch_size):

            #x = input_nn_train[i:i + batch_size, :,:]
            #y = output_nn_train[i:i + batch_size, :,:]
            #x, y = np_to_tensor(t.batch())
            #model.forward(x)  # run forward pass to initialize weights
            
            with tf.GradientTape() as test_tape:
                # test_tape.watch(model.trainable_variables)
                # Step 5
                with tf.GradientTape() as train_tape:
                    train_loss, _ = compute_loss(model, x, y)

                # Step 6
                gradients = train_tape.gradient(train_loss, model.trainable_variables)
                k = 0
                
                model_copy = copy_model(model, x)

                for j in range(len(model_copy.layers)):
                    
#                     if isinstance(model_copy.layers[j], tf.keras.layers.RNN):

#                         #print(j+1, 'LMU')
                        
#                         model_copy.layers[j].cell.kernel = tf.subtract(model.layers[j].cell.kernel,
#                                     tf.multiply(lr_inner, gradients[k]))
#                         model_copy.layers[j].cell.thr = tf.subtract(model.layers[j].cell.thr,
#                                 tf.multiply(lr_inner, gradients[k+1]))
#                         model_copy.layers[j].cell.hidden_cell.kernel = tf.subtract(model.layers[j].cell.hidden_cell.kernel,
#                                     tf.multiply(lr_inner, gradients[k+2]))
#                         model_copy.layers[j].cell.hidden_cell.recurrent_kernel = tf.subtract(model.layers[j].cell.hidden_cell.recurrent_kernel,
#                                     tf.multiply(lr_inner, gradients[k+3]))
#                         model_copy.layers[j].cell.hidden_cell.bias = tf.subtract(model.layers[j].cell.hidden_cell.bias,
#                                     tf.multiply(lr_inner, gradients[k+4]))

#                         k += 5
                        
#                     if isinstance(model_copy.layers[j], tf.keras.layers.LSTM):
                        
#                         model_copy.layers[j].cell.kernel = tf.subtract(model.layers[j].cell.kernel,
#                                     tf.multiply(lr_inner, gradients[k]))
#                         model_copy.layers[j].cell.recurrent_kernel = tf.subtract(model.layers[j].cell.recurrent_kernel,
#                                 tf.multiply(lr_inner, gradients[k+1]))
#                         model_copy.layers[j].cell.bias = tf.subtract(model.layers[j].cell.bias,
#                                     tf.multiply(lr_inner, gradients[k+2]))
#                         k += 3
                        
                        
                    if isinstance(model_copy.layers[j], tf.keras.layers.RNN):
                        
                        model_copy.layers[j].cell.input_weights = tf.subtract(model.layers[j].cell.input_weights,
                                    tf.multiply(lr_inner, gradients[k]))
                        model_copy.layers[j].cell.recurrent_weights = tf.subtract(model.layers[j].cell.recurrent_weights,
                                tf.multiply(lr_inner, gradients[k+1]))
#                         model_copy.layers[j].cell.bias = tf.subtract(model.layers[j].cell.bias,
#                                     tf.multiply(lr_inner, gradients[k+2]))
                        k += 2
                        
                    elif isinstance(model_copy.layers[j], tf.keras.layers.Dense):
                        
                        #print(j+1, 'Dense layer')
                        
                        model_copy.layers[j].kernel = tf.subtract(model.layers[j].kernel,
                                tf.multiply(lr_inner, gradients[k]))
                        model_copy.layers[j].bias = tf.subtract(model.layers[j].bias,
                                tf.multiply(lr_inner, gradients[k+1]))
                        k += 2
                        
                    

                # Step 8
                test_loss, logits = compute_loss(model_copy, input_nn_test, output_nn_test)

            # Step 8
            gradients_ = test_tape.gradient(test_loss, model.trainable_variables)
            
            meta_grads = [g + mg for g, mg in zip(gradients_, meta_grads)]
            
            
        

            # Logs
            total_loss += test_loss
            loss = total_loss / (i+1.0)
            losses.append(loss)

    #         if loss.numpy()<1e-2:
    #             break
        print('Loss: '+str(np.linalg.norm(loss)), 'Epoch nr: '+str(tt+1))
        optimizer.apply_gradients(zip(meta_grads, model.trainable_variables))
        if np.linalg.norm(loss)<1e-3:
            break
                #start = time.time()
                
    return loss


def MAML_LMU_multiple(model, train_dataset, input_nn_test, output_nn_test, epochs, inner_epochs):

    lr_inner = 0.001
    for tt in range(epochs):

        total_loss = 0
        losses = []
        #start = time.time()
        meta_grads = [tf.zeros_like(w) for w in model.trainable_variables]
        # Step 3 and 4
        for i, (x, y) in enumerate(train_dataset):
        #for i in range(0, len(input_nn_train), batch_size):

            #x = input_nn_train[i:i + batch_size, :,:]
            #y = output_nn_train[i:i + batch_size, :,:]
            #x, y = np_to_tensor(t.batch())
            #model.forward(x)  # run forward pass to initialize weights
            
            with tf.GradientTape() as test_tape:
                # test_tape.watch(model.trainable_variables)
                # Step 5

                for ee in range(inner_epochs):
                    with tf.GradientTape(persistent=True) as train_tape:
                        train_loss, _ = compute_loss(model, x, y)
    
                    # Step 6
                    gradients = train_tape.gradient(train_loss, model.trainable_variables)
                    k = 0
                    
                    model_copy = copy_model(model, x)
    
                    for j in range(len(model_copy.layers)):
                        
    #                     if isinstance(model_copy.layers[j], tf.keras.layers.RNN):
    
    #                         #print(j+1, 'LMU')
                            
    #                         model_copy.layers[j].cell.kernel = tf.subtract(model.layers[j].cell.kernel,
    #                                     tf.multiply(lr_inner, gradients[k]))
    #                         model_copy.layers[j].cell.thr = tf.subtract(model.layers[j].cell.thr,
    #                                 tf.multiply(lr_inner, gradients[k+1]))
    #                         model_copy.layers[j].cell.hidden_cell.kernel = tf.subtract(model.layers[j].cell.hidden_cell.kernel,
    #                                     tf.multiply(lr_inner, gradients[k+2]))
    #                         model_copy.layers[j].cell.hidden_cell.recurrent_kernel = tf.subtract(model.layers[j].cell.hidden_cell.recurrent_kernel,
    #                                     tf.multiply(lr_inner, gradients[k+3]))
    #                         model_copy.layers[j].cell.hidden_cell.bias = tf.subtract(model.layers[j].cell.hidden_cell.bias,
    #                                     tf.multiply(lr_inner, gradients[k+4]))
    
    #                         k += 5
                            
    #                     if isinstance(model_copy.layers[j], tf.keras.layers.LSTM):
                            
    #                         model_copy.layers[j].cell.kernel = tf.subtract(model.layers[j].cell.kernel,
    #                                     tf.multiply(lr_inner, gradients[k]))
    #                         model_copy.layers[j].cell.recurrent_kernel = tf.subtract(model.layers[j].cell.recurrent_kernel,
    #                                 tf.multiply(lr_inner, gradients[k+1]))
    #                         model_copy.layers[j].cell.bias = tf.subtract(model.layers[j].cell.bias,
    #                                     tf.multiply(lr_inner, gradients[k+2]))
    #                         k += 3
                            
                            
                        if isinstance(model_copy.layers[j], tf.keras.layers.RNN):
                            
                            model_copy.layers[j].cell.input_weights = tf.subtract(model.layers[j].cell.input_weights,
                                        tf.multiply(lr_inner, gradients[k]))
                            model_copy.layers[j].cell.recurrent_weights = tf.subtract(model.layers[j].cell.recurrent_weights,
                                    tf.multiply(lr_inner, gradients[k+1]))
    #                         model_copy.layers[j].cell.bias = tf.subtract(model.layers[j].cell.bias,
    #                                     tf.multiply(lr_inner, gradients[k+2]))
                            k += 2
                            
                        elif isinstance(model_copy.layers[j], tf.keras.layers.Dense):
                            
                            #print(j+1, 'Dense layer')
                            
                            model_copy.layers[j].kernel = tf.subtract(model.layers[j].kernel,
                                    tf.multiply(lr_inner, gradients[k]))
                            model_copy.layers[j].bias = tf.subtract(model.layers[j].bias,
                                    tf.multiply(lr_inner, gradients[k+1]))
                            k += 2
                            
                        

                # Step 8
                test_loss, logits = compute_loss(model_copy, input_nn_test, output_nn_test)

            # Step 8
            gradients_ = test_tape.gradient(test_loss, model.trainable_variables)
            
            meta_grads = [g + mg for g, mg in zip(gradients_, meta_grads)]
            
            
        

            # Logs
            total_loss += test_loss
            loss = total_loss / (i+1.0)
            losses.append(loss)

    #         if loss.numpy()<1e-2:
    #             break
        print('Loss: '+str(np.linalg.norm(loss)), 'Epoch nr: '+str(tt+1))
        optimizer.apply_gradients(zip(meta_grads, model.trainable_variables))
        if np.linalg.norm(loss)<6e-4:
            break
                #start = time.time()
                
    return loss



In [24]:
loss = MAML_LMU_multiple(model, train_dataset, input_nn_test[:,:,:], output_nn_test[:,:,:], 1000, 2)

Loss: 0.44406763 Epoch nr: 1
Loss: 0.23400532 Epoch nr: 2
Loss: 0.16629691 Epoch nr: 3
Loss: 0.085909575 Epoch nr: 4
Loss: 0.05207492 Epoch nr: 5
Loss: 0.04246419 Epoch nr: 6
Loss: 0.06850448 Epoch nr: 7
Loss: 0.076787196 Epoch nr: 8
Loss: 0.01738607 Epoch nr: 9
Loss: 0.21990865 Epoch nr: 10
Loss: 0.080769025 Epoch nr: 11
Loss: 0.012598131 Epoch nr: 12
Loss: 0.00016203155 Epoch nr: 13


In [ ]:
aus_klassisches_trainiertes_model = model2.predict(input_nn_test[0:1,:,:])

In [ ]:
aus_meta_trainiertes_model   = model.forward(input_nn_test[0:1,:,:])

In [ ]:
aus_meta_trainiertes_model

In [ ]:
plt.plot(input_nn_test[0], aus_klassisches_trainiertes_model.reshape(1*62, 6), label='klassisch', color='blue')
plt.plot(input_nn_test[0],aus_meta_trainiertes_model.numpy().reshape(1*62, 6), label='meta', color='orange')
plt.scatter(input_nn_test[0],output_nn_test[0], label='Data', color='green')
plt.legend()

In [25]:
#klassiches_pretrainierte_model = model2.get_weights()

In [35]:
model2.set_weights(klassiches_pretrainierte_model)

In [36]:
callbacks = [
                 EarlyStoppingByAccuracy(monitor='loss', value=5e-4, verbose=1)
            ]

In [37]:
History = model2.fit(input_nn_validation_inp[:,:,:], input_nn_validation_output[:,:,:], batch_size=8, epochs=1000, callbacks=callbacks)

Epoch 1/1000
1/1 [==============================] - 1s 1s/step - loss: 9.3252e-04
Epoch 2/1000
1/1 [==============================] - 1s 1s/step - loss: 9.6497e-04
Epoch 3/1000
1/1 [==============================] - 1s 1s/step - loss: 0.0012
Epoch 4/1000
1/1 [==============================] - 1s 1s/step - loss: 0.0012
Epoch 5/1000
1/1 [==============================] - 1s 1s/step - loss: 0.0013
Epoch 6/1000
1/1 [==============================] - 1s 1s/step - loss: 0.0013
Epoch 7/1000
1/1 [==============================] - 1s 1s/step - loss: 0.0013
Epoch 8/1000
1/1 [==============================] - 1s 1s/step - loss: 0.0014
Epoch 9/1000
1/1 [==============================] - 1s 1s/step - loss: 0.0013
Epoch 10/1000
1/1 [==============================] - 1s 1s/step - loss: 0.0011
Epoch 11/1000
1/1 [==============================] - 1s 1s/step - loss: 0.0012
Epoch 12/1000
1/1 [==============================] - 1s 1s/step - loss: 0.0015
Epoch 13/1000
1/1 [==============================] - 

In [30]:
model2.set_weights(model.get_weights())

In [31]:
History_1 = model2.fit(input_nn_validation_inp[:,:,:], input_nn_validation_output[:,:,:], batch_size=8, epochs=1000, callbacks=callbacks)

Epoch 1/1000
1/1 [==============================] - 1s 1s/step - loss: 0.0170
Epoch 2/1000
1/1 [==============================] - 1s 1s/step - loss: 0.5382
Epoch 3/1000
1/1 [==============================] - 1s 1s/step - loss: 0.0127
Epoch 4/1000
1/1 [==============================] - 1s 1s/step - loss: 0.4921
Epoch 5/1000
1/1 [==============================] - 1s 1s/step - loss: 0.3005
Epoch 6/1000
1/1 [==============================] - 1s 1s/step - loss: 0.0123
Epoch 7/1000
1/1 [==============================] - 1s 1s/step - loss: 0.2158
Epoch 8/1000
1/1 [==============================] - 1s 1s/step - loss: 0.1211
Epoch 9/1000
1/1 [==============================] - 1s 1s/step - loss: 0.0031
Epoch 10/1000
1/1 [==============================] - 1s 1s/step - loss: 0.0703
Epoch 11/1000
1/1 [==============================] - 1s 1s/step - loss: 0.0896
Epoch 12/1000
1/1 [==============================] - 1s 1s/step - loss: 0.0163
Epoch 13/1000
1/1 [==============================] - 1s 1s/st

In [51]:
plt.rcParams.update({'font.size': 11})
plt.plot(History.history['loss'], label='Klassische Trainiertes Model')
plt.plot(History_1.history['loss'], label='Meta Trainiertes Model')
plt.xlabel('Self-learning steps')
plt.ylabel('Validation loss')
plt.ylim(0, 0.008)
plt.legend()